In [ ]:
import sys
sys.path.insert(0, '..')

from config import Config
from app.models import init_db, insert_alert, get_alerts, get_stats

In [ ]:
conn = init_db(Config.DB_PATH)
conn.execute('DELETE FROM alerts')
conn.commit()
print('DB initialized and cleared')

In [ ]:
import uuid
from datetime import datetime, timedelta

samples = [
    {
        'alert_id': str(uuid.uuid4()),
        'timestamp': (datetime.utcnow() - timedelta(days=i)).isoformat(),
        'raw_text': text,
        'tactic_category': tactic,
        'confidence_score': 0.75 + i * 0.05,
        'is_duplicate': 1 if i % 2 == 0 else 0,
    }
    for i, (text, tactic) in enumerate([
        ('Detected outbound traffic to known C2 IP over port 443', 'command-and-control'),
        ('Multiple failed login attempts from external IP 10.45.2.1', 'initial-access'),
        ('Suspicious PowerShell execution detected on host srv-015', 'execution'),
        ('Registry persistence key added under HKLM\\Software\\Microsoft\\Windows\\CurrentVersion\\Run', 'persistence'),
        ('Large volume of outbound FTP traffic from ws-088 to external IP 192.168.99.1', 'exfiltration'),
    ])
]

for alert in samples:
    insert_alert(conn, alert)

print(f'Inserted {len(samples)} sample alerts')

In [ ]:
alerts = get_alerts(conn, limit=10, offset=0)
for a in alerts:
    print(f"{a['alert_id'][:8]}... | {a['tactic_category']:22s} | dup={a['is_duplicate']} | {a['raw_text'][:60]}")

In [ ]:
stats = get_stats(conn)
print('Tactic distribution:', stats['tactic_distribution'])
print(f"Duplicate rate: {stats['duplicate_rate']:.2%}")
print('Daily volume:', stats['daily_volume'])

In [ ]:
conn.close()
print('Connection closed')